In [2]:
import numpy as np 

ACTIONS = [
    "ALLOW", 
    "MONITOR",
    "RATE_LIMIT", 
    "BLOCK", 
    "ESCALATE"
]

ACTION_TO_ID = {
    action:index 
    for index,action in enumerate(ACTIONS) 
}
ID_TO_ACTION = {
    index:action 
    for action, index in ACTION_TO_ID.items()
}

print(ACTION_TO_ID)

{'ALLOW': 0, 'MONITOR': 1, 'RATE_LIMIT': 2, 'BLOCK': 3, 'ESCALATE': 4}


In [3]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path(
    r"D:\Python project\PROJECT"
)

MODELS_DIR = PROJECT_ROOT / "models"

label_encoder = joblib.load(
    MODELS_DIR / "xgboost_label_encoder.joblib"
)

print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)

Number of classes: 7
Classes: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']


In [5]:
def build_state(
    result,
    previous_action="ALLOW",
    recent_alert_density=0.0,
    action_cooldown=0.0
):
    anomaly_score = float(
        result["anomaly_score"]
    )

    anomaly_score_normalized = np.clip(
        np.log1p(anomaly_score) / np.log1p(100.0),
        0.0,
        1.0
    )

    attack_probabilities = np.array([
        result["attack_probabilities"][label]
        for label in label_encoder.classes_
    ], dtype=np.float32)

    previous_action_id = ACTION_TO_ID[
        previous_action
    ]

    previous_action_normalized = (
        previous_action_id / (len(ACTIONS) - 1)
    )

    state = np.concatenate([
        np.array(
            [anomaly_score_normalized],
            dtype=np.float32
        ),
        attack_probabilities,
        np.array(
            [
                previous_action_normalized,
                recent_alert_density,
                action_cooldown
            ],
            dtype=np.float32
        )
    ])

    assert state.shape == (11,)

    return state

In [7]:
# load lai kien truc detection 
import json 
import joblib 
import pandas as pd 
import torch 
from torch import nn 
from xgboost import XGBClassifier 

device = torch.device(
    "cuda" if torch.cuda.is_available else "cpu "
)
print(device)

cuda


In [8]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [9]:
ae_checkpoint = torch.load(
    MODELS_DIR / "autoencoder_full.pt",
    map_location=device,
    weights_only=False
)

ae_model = Autoencoder(
    ae_checkpoint["input_dim"]
).to(device)

ae_model.load_state_dict(
    ae_checkpoint["model_state_dict"]
)

ae_model.eval()

scaler = joblib.load(
    MODELS_DIR / "autoencoder_scaler.joblib"
)

with open(
    MODELS_DIR / "anomaly_threshold.json",
    "r"
) as f:
    ae_metadata = json.load(f)

ae_threshold = ae_metadata["threshold"]
ae_feature_cols = ae_metadata["feature_columns"]

xgb_model = XGBClassifier()

xgb_model.load_model(
    MODELS_DIR / "xgboost_classifier.json"
)

label_encoder = joblib.load(
    MODELS_DIR / "xgboost_label_encoder.joblib"
)

with open(
    MODELS_DIR / "xgboost_metadata.json",
    "r"
) as f:
    xgb_metadata = json.load(f)

xgb_feature_cols = xgb_metadata["feature_columns"]

assert ae_feature_cols == xgb_feature_cols

feature_cols = xgb_feature_cols

print("Autoencoder loaded")
print("XGBoost loaded")
print("Features:", len(feature_cols))
print("Classes:", label_encoder.classes_)

Autoencoder loaded
XGBoost loaded
Features: 52
Classes: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']


In [10]:
def analyze_flow(flow_df, true_label=None):
    x_raw = flow_df[feature_cols]

    # Autoencoder dùng dữ liệu đã scale
    x_scaled = scaler.transform(
        x_raw
    ).astype("float32")

    x_tensor = torch.tensor(
        x_scaled,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        reconstructed = ae_model(x_tensor)

    anomaly_score = torch.mean(
        (x_tensor - reconstructed) ** 2,
        dim=1
    ).item()

    is_anomaly = anomaly_score > ae_threshold

    # XGBoost dùng dữ liệu raw
    predicted_id = int(
        xgb_model.predict(x_raw)[0]
    )

    probabilities = (
        xgb_model.predict_proba(x_raw)[0]
    )

    predicted_label = label_encoder.inverse_transform(
        [predicted_id]
    )[0]

    result = {
        "true_label": true_label,
        "anomaly_score": anomaly_score,
        "is_anomaly": is_anomaly,
        "predicted_attack": predicted_label,
        "confidence": float(
            probabilities[predicted_id]
        ),
        "attack_probabilities": {
            label: float(probability)
            for label, probability in zip(
                label_encoder.classes_,
                probabilities
            )
        }
    }

    return result

In [11]:
test_df = pd.read_csv(
    PROJECT_ROOT / "data" / "splits" / "test.csv",
    low_memory=False
)

sample_idx = test_df.index[
    test_df["Attack Type"] == "DoS"
][0]

one_flow = test_df.loc[
    [sample_idx],
    feature_cols
]

true_label = test_df.loc[
    sample_idx,
    "Attack Type"
]

result = analyze_flow(
    one_flow,
    true_label=true_label
)

print(result)

{'true_label': 'DoS', 'anomaly_score': 2.819601535797119, 'is_anomaly': True, 'predicted_attack': 'DoS', 'confidence': 0.9999188184738159, 'attack_probabilities': {'Bots': 7.798544174875133e-07, 'Brute Force': 9.999748726841062e-07, 'DDoS': 9.248878086509649e-06, 'DoS': 0.9999188184738159, 'Normal Traffic': 6.305109855020419e-05, 'Port Scanning': 6.737575859006029e-06, 'Web Attacks': 3.789530751419079e-07}}


In [12]:
state = build_state(
    result,
    previous_action="ALLOW",
    recent_alert_density=0.0,
    action_cooldown=0.0
)

print("State:", state)
print("State shape:", state.shape)

State: [2.9038161e-01 7.7985442e-07 9.9997487e-07 9.2488781e-06 9.9991882e-01
 6.3051099e-05 6.7375759e-06 3.7895308e-07 0.0000000e+00 0.0000000e+00
 0.0000000e+00]
State shape: (11,)


In [13]:
RISK_LEVELS = {
    "Normal Traffic": "normal",
    "DoS": "high",
    "DDoS": "high",
    "Port Scanning": "low",
    "Brute Force": "low",
    "Bots": "low",
    "Web Attacks": "low"
}

In [14]:
REWARD_TABLE = {
    "normal": {
        "ALLOW": 5,
        "MONITOR": 3,
        "RATE_LIMIT": -5,
        "BLOCK": -30,
        "ESCALATE": -8
    },

    "low": {
        "ALLOW": -8,
        "MONITOR": 3,
        "RATE_LIMIT": 6,
        "BLOCK": 4,
        "ESCALATE": 5
    },

    "high": {
        "ALLOW": -30,
        "MONITOR": -5,
        "RATE_LIMIT": 5,
        "BLOCK": 15,
        "ESCALATE": 12
    }
}

In [15]:
def compute_reward(true_label, action_id):
    action_name = ID_TO_ACTION[action_id]
    risk_level = RISK_LEVELS[true_label]

    return REWARD_TABLE[risk_level][action_name]

In [16]:
for label in [
    "Normal Traffic",
    "DoS",
    "Bots"
]:
    print(
        label,
        [
            compute_reward(label, action_id)
            for action_id in range(len(ACTIONS))
        ]
    )

Normal Traffic [5, 3, -5, -30, -8]
DoS [-30, -5, 5, 15, 12]
Bots [-8, 3, 6, 4, 5]


In [17]:
import gymnasium as gym
from gymnasium import spaces

demo_df = test_df.iloc[:20].copy()

flow_results = []

for idx, row in demo_df.iterrows():
    one_flow = test_df.loc[
        [idx],
        feature_cols
    ]

    flow_result = analyze_flow(
        one_flow,
        true_label=row["Attack Type"]
    )

    flow_results.append(flow_result)

print("Number of flows:", len(flow_results))

Number of flows: 20


In [18]:
class NetworkDefenseEnv(gym.Env):
    def __init__(self, flow_results):
        super().__init__()

        self.flow_results = flow_results

        self.action_space = spaces.Discrete(
            len(ACTIONS)
        )

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(11,),
            dtype=np.float32
        )

        self.max_cooldown = 3
        self.reset()

    def _get_observation(self):
        if self.current_step >= len(self.flow_results):
            return np.zeros(
                11,
                dtype=np.float32
            )

        current_result = self.flow_results[
            self.current_step
        ]

        if self.recent_alerts:
            recent_alert_density = np.mean(
                self.recent_alerts[-10:]
            )
        else:
            recent_alert_density = 0.0

        cooldown_normalized = np.clip(
            self.action_cooldown / self.max_cooldown,
            0.0,
            1.0
        )

        return build_state(
            current_result,
            previous_action=self.previous_action,
            recent_alert_density=recent_alert_density,
            action_cooldown=cooldown_normalized
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.previous_action = "ALLOW"
        self.recent_alerts = []
        self.action_cooldown = 0

        observation = self._get_observation()

        return observation, {}

    def step(self, action):
        action = int(action)

        if not self.action_space.contains(action):
            raise ValueError("Invalid action")

        current_result = self.flow_results[
            self.current_step
        ]

        action_name = ID_TO_ACTION[action]

        reward = compute_reward(
            current_result["true_label"],
            action
        )

        detector_alert = (
            current_result["is_anomaly"]
            or current_result["predicted_attack"]
            != "Normal Traffic"
        )

        self.recent_alerts.append(
            int(detector_alert)
        )

        if action_name == "BLOCK":
            self.action_cooldown = 3

        elif action_name == "RATE_LIMIT":
            self.action_cooldown = 2

        elif action_name == "ESCALATE":
            self.action_cooldown = 1

        else:
            self.action_cooldown = max(
                0,
                self.action_cooldown - 1
            )

        self.previous_action = action_name
        self.current_step += 1

        terminated = (
            self.current_step
            >= len(self.flow_results)
        )

        observation = self._get_observation()

        info = {
            "action": action_name,
            "true_label": current_result["true_label"],
            "predicted_attack": current_result[
                "predicted_attack"
            ],
            "reward": reward
        }

        return (
            observation,
            float(reward),
            terminated,
            False,
            info
        )

In [19]:
env = NetworkDefenseEnv(flow_results)

observation, info = env.reset()

print("Initial observation:", observation)
print("Observation shape:", observation.shape)

Initial observation: [2.10351849e-04 2.15204864e-07 3.17707503e-07 3.40224733e-06
 7.18567981e-06 9.99986291e-01 2.45778824e-06 1.01713766e-07
 0.00000000e+00 0.00000000e+00 0.00000000e+00]
Observation shape: (11,)


In [20]:
observation, reward, terminated, truncated, info = (
    env.step(ACTION_TO_ID["BLOCK"])
)

print("Reward:", reward)
print("Next observation:", observation)
print("Info:", info)

Reward: -30.0
Next observation: [7.4041214e-05 2.0619134e-07 3.2725029e-07 3.5449602e-06 7.2636376e-06
 9.9998605e-01 2.5608867e-06 1.0250015e-07 7.5000000e-01 0.0000000e+00
 1.0000000e+00]
Info: {'action': 'BLOCK', 'true_label': 'Normal Traffic', 'predicted_attack': 'Normal Traffic', 'reward': -30}


In [21]:
HIGH_RISK_ATTACKS = {
    "DoS",
    "DDoS"
}

def rule_based_policy(result):
    predicted_attack = result["predicted_attack"]
    confidence = result["confidence"]
    anomaly_score = result["anomaly_score"]
    is_anomaly = result["is_anomaly"]

    # Model không chắc chắn nhưng anomaly cao
    if is_anomaly and confidence < 0.70:
        return ACTION_TO_ID["ESCALATE"]

    # Attack nguy hiểm và anomaly cao
    if (
        predicted_attack in HIGH_RISK_ATTACKS
        and anomaly_score > ae_threshold
    ):
        return ACTION_TO_ID["BLOCK"]

    # Attack nhưng chưa đủ chắc để block
    if (
        predicted_attack != "Normal Traffic"
        and confidence >= 0.70
    ):
        return ACTION_TO_ID["RATE_LIMIT"]

    # Bất thường nhưng chưa phân loại rõ
    if is_anomaly:
        return ACTION_TO_ID["MONITOR"]

    return ACTION_TO_ID["ALLOW"]

In [22]:
def run_policy(env, policy):
    observation, info = env.reset()

    records = []
    total_reward = 0.0

    while True:
        current_result = env.flow_results[
            env.current_step
        ]

        action_id = policy(current_result)

        (
            observation,
            reward,
            terminated,
            truncated,
            info
        ) = env.step(action_id)

        records.append({
            "true_label": info["true_label"],
            "predicted_attack": info[
                "predicted_attack"
            ],
            "action": info["action"],
            "reward": reward
        })

        total_reward += reward

        if terminated or truncated:
            break

    return pd.DataFrame(records), total_reward

In [23]:
rule_results, rule_total_reward = run_policy(
    env,
    rule_based_policy
)

print("Total reward:", rule_total_reward)
print(
    "Average reward:",
    rule_results["reward"].mean()
)

display(
    rule_results["action"].value_counts()
)

display(rule_results)

Total reward: 120.0
Average reward: 6.0


action
ALLOW    18
BLOCK     2
Name: count, dtype: int64

,true_label,predicted_attack,action,reward
0,Normal Traffic,Normal Traffic,ALLOW,5.0
1,Normal Traffic,Normal Traffic,ALLOW,5.0
2,Normal Traffic,Normal Traffic,ALLOW,5.0
3,Normal Traffic,Normal Traffic,ALLOW,5.0
4,Normal Traffic,Normal Traffic,ALLOW,5.0
5,Normal Traffic,Normal Traffic,ALLOW,5.0
6,Normal Traffic,Normal Traffic,ALLOW,5.0
7,Normal Traffic,Normal Traffic,ALLOW,5.0
8,Normal Traffic,Normal Traffic,ALLOW,5.0
9,Normal Traffic,Normal Traffic,ALLOW,5.0


In [24]:
evaluation_df = test_df.iloc[:1000].copy()

evaluation_results = []

for idx, row in evaluation_df.iterrows():
    one_flow = test_df.loc[
        [idx],
        feature_cols
    ]

    flow_result = analyze_flow(
        one_flow,
        true_label=row["Attack Type"]
    )

    evaluation_results.append(flow_result)

print(
    "Number of evaluated flows:",
    len(evaluation_results)
)

Number of evaluated flows: 1000


In [25]:
evaluation_env = NetworkDefenseEnv(
    evaluation_results
)

rule_results_1000, rule_total_reward_1000 = (
    run_policy(
        evaluation_env,
        rule_based_policy
    )
)

print(
    "Total reward:",
    rule_total_reward_1000
)

print(
    "Average reward:",
    rule_results_1000["reward"].mean()
)

print(
    "Action distribution:"
)

display(
    rule_results_1000["action"].value_counts()
)

Total reward: 5753.0
Average reward: 5.753
Action distribution:


action
ALLOW         847
BLOCK          73
RATE_LIMIT     73
MONITOR         7
Name: count, dtype: int64

In [26]:
normal_mask = (
    rule_results_1000["true_label"]
    == "Normal Traffic"
)

attack_mask = ~normal_mask

actions = rule_results_1000["action"]

missed_attack = (
    attack_mask
    & actions.eq("ALLOW")
)

false_mitigation = (
    normal_mask
    & actions.isin([
        "RATE_LIMIT",
        "BLOCK",
        "ESCALATE"
    ])
)

block_on_attack = (
    attack_mask
    & actions.eq("BLOCK")
)

metrics = {
    "normal_flows": int(normal_mask.sum()),
    "attack_flows": int(attack_mask.sum()),
    "missed_attacks": int(missed_attack.sum()),
    "missed_attack_rate": float(
        missed_attack.sum() / attack_mask.sum()
    ),
    "false_mitigations": int(
        false_mitigation.sum()
    ),
    "false_mitigation_rate": float(
        false_mitigation.sum() / normal_mask.sum()
    ),
    "block_rate_on_attack": float(
        block_on_attack.sum() / attack_mask.sum()
    ),
    "escalation_rate": float(
        actions.eq("ESCALATE").mean()
    )
}

print(metrics)

{'normal_flows': 854, 'attack_flows': 146, 'missed_attacks': 0, 'missed_attack_rate': 0.0, 'false_mitigations': 0, 'false_mitigation_rate': 0.0, 'block_rate_on_attack': 0.5, 'escalation_rate': 0.0}


In [27]:
display(
    rule_results_1000["true_label"].value_counts()
)

true_label
Normal Traffic    854
DoS                58
DDoS               51
Port Scanning      31
Brute Force         3
Bots                2
Web Attacks         1
Name: count, dtype: int64

In [28]:
action_by_label = pd.crosstab(
    rule_results_1000["true_label"],
    rule_results_1000["action"]
)

display(action_by_label)

action,ALLOW,BLOCK,MONITOR,RATE_LIMIT
true_label,,,,
Bots,0,0,0,2
Brute Force,0,0,0,3
DDoS,0,28,0,23
DoS,0,45,0,13
Normal Traffic,847,0,7,0
Port Scanning,0,0,0,31
Web Attacks,0,0,0,1


In [29]:
evaluation_parts = []

for label, group in test_df.groupby("Attack Type"):
    evaluation_parts.append(
        group.sample(
            n=min(len(group), 200),
            random_state=42
        )
    )

evaluation_df = (
    pd.concat(evaluation_parts)
    .reset_index(drop=True)
)

print(evaluation_df["Attack Type"].value_counts())

Attack Type
Bots              200
Brute Force       200
DDoS              200
DoS               200
Normal Traffic    200
Port Scanning     200
Web Attacks       200
Name: count, dtype: int64


In [30]:
train_df = pd.read_csv(
    PROJECT_ROOT / "data" / "splits" / "train.csv",
    low_memory=False
)

train_demo_df = train_df.iloc[:1000].copy()

print(train_demo_df["Attack Type"].value_counts())

Attack Type
Normal Traffic    841
DoS                70
DDoS               48
Port Scanning      35
Web Attacks         4
Bots                1
Brute Force         1
Name: count, dtype: int64


In [31]:
train_results = []

for idx, row in train_demo_df.iterrows():
    one_flow = train_demo_df.loc[
        [idx],
        feature_cols
    ]

    flow_result = analyze_flow(
        one_flow,
        true_label=row["Attack Type"]
    )

    train_results.append(flow_result)

print(
    "Number of training flows:",
    len(train_results)
)

Number of training flows: 1000


In [32]:
train_env = NetworkDefenseEnv(
    train_results
)

observation, info = train_env.reset()

print("Observation shape:", observation.shape)
print("Initial observation:", observation)

Observation shape: (11,)
Initial observation: [1.7720343e-03 3.6139716e-07 9.2501881e-07 3.6907252e-06 6.9619850e-06
 9.9998271e-01 4.8856878e-06 4.0521488e-07 0.0000000e+00 0.0000000e+00
 0.0000000e+00]


In [33]:
from stable_baselines3 import DQN

dqn_model = DQN(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=1e-3,
    buffer_size=5_000,
    learning_starts=100,
    batch_size=64,
    gamma=0.95,
    exploration_fraction=0.20,
    exploration_final_eps=0.05,
    target_update_interval=250,
    verbose=1,
    seed=42,
    device="cuda"
)

print("DQN device:", dqn_model.device)

dqn_model.learn(
    total_timesteps=10_000,
    progress_bar=True
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Output()

DQN device: cuda


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1e+03    |
|    ep_rew_mean      | 2.52e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 773      |
|    time_elapsed     | 5        |
|    total_timesteps  | 4000     |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.33     |
|    n_updates        | 974      |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1e+03    |
|    ep_rew_mean      | 4.02e+03 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 768      |
|    time_elapsed     | 10       |
|    total_timesteps  | 8000     |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 1.12     |
|    n_updates        | 1974     |
----------------------------------


In [34]:
import pandas as pd

def run_dqn_policy(env, model):
    observation, _ = env.reset()
    rows = []
    total_reward = 0.0
    terminated = False
    truncated = False

    for result in env.flow_results:
        action_id, _ = model.predict(
            observation,
            deterministic=True
        )
        action_id = int(action_id)

        observation, reward, terminated, truncated, _ = env.step(
            action_id
        )

        rows.append({
            "true_label": result["true_label"],
            "predicted_attack": result["predicted_attack"],
            "confidence": result["confidence"],
            "action": ID_TO_ACTION[action_id],
            "reward": reward
        })

        total_reward += reward

        if terminated or truncated:
            break

    return pd.DataFrame(rows), total_reward


dqn_evaluation_env = NetworkDefenseEnv(
    evaluation_results
)

dqn_results_1000, dqn_total_reward_1000 = run_dqn_policy(
    dqn_evaluation_env,
    dqn_model
)

print("Total reward:", dqn_total_reward_1000)
print("Average reward:", dqn_results_1000["reward"].mean())

print("\nAction distribution:")
display(
    dqn_results_1000["action"]
    .value_counts()
    .reindex(ACTIONS, fill_value=0)
)

display(dqn_results_1000.head(20))

Total reward: 6108.0
Average reward: 6.108

Action distribution:


action
ALLOW         854
MONITOR         5
RATE_LIMIT     31
BLOCK         108
ESCALATE        2
Name: count, dtype: int64

,true_label,predicted_attack,confidence,action,reward
0,Normal Traffic,Normal Traffic,0.999986,ALLOW,5.0
1,Normal Traffic,Normal Traffic,0.999986,ALLOW,5.0
2,Normal Traffic,Normal Traffic,0.999974,ALLOW,5.0
3,Normal Traffic,Normal Traffic,0.999946,ALLOW,5.0
4,Normal Traffic,Normal Traffic,0.999972,ALLOW,5.0
5,Normal Traffic,Normal Traffic,0.999986,ALLOW,5.0
6,Normal Traffic,Normal Traffic,0.999986,ALLOW,5.0
7,Normal Traffic,Normal Traffic,0.999940,ALLOW,5.0
8,Normal Traffic,Normal Traffic,0.999986,ALLOW,5.0
9,Normal Traffic,Normal Traffic,0.999987,ALLOW,5.0


In [35]:
comparison_df = pd.DataFrame({
    "Policy": ["Rule-based", "DQN"],
    "Total reward": [
        rule_total_reward_1000,
        dqn_total_reward_1000
    ],
    "Average reward": [
        rule_results_1000["reward"].mean(),
        dqn_results_1000["reward"].mean()
    ]
})

display(comparison_df)

,Policy,Total reward,Average reward
0,Rule-based,5753.0,5.753
1,DQN,6108.0,6.108


In [36]:
def calculate_defense_metrics(results_df):
    is_attack = results_df["true_label"] != "Normal Traffic"

    attack_results = results_df[is_attack]
    normal_results = results_df[~is_attack]

    missed_attacks = (
        attack_results["action"] == "ALLOW"
    ).sum()

    false_mitigations = (
        normal_results["action"] != "ALLOW"
    ).sum()

    return {
        "normal_flows": len(normal_results),
        "attack_flows": len(attack_results),
        "missed_attacks": missed_attacks,
        "missed_attack_rate": (
            missed_attacks / len(attack_results)
        ),
        "false_mitigations": false_mitigations,
        "false_mitigation_rate": (
            false_mitigations / len(normal_results)
        ),
        "block_rate_on_attack": (
            attack_results["action"] == "BLOCK"
        ).mean(),
        "escalation_rate": (
            results_df["action"] == "ESCALATE"
        ).mean()
    }


rule_metrics = calculate_defense_metrics(rule_results_1000)
dqn_metrics = calculate_defense_metrics(dqn_results_1000)

metrics_comparison = pd.DataFrame(
    [rule_metrics, dqn_metrics],
    index=["Rule-based", "DQN"]
)

display(metrics_comparison)

print("DQN: true label × action")
display(
    pd.crosstab(
        dqn_results_1000["true_label"],
        dqn_results_1000["action"]
    )
)

,normal_flows,attack_flows,missed_attacks,missed_attack_rate,false_mitigations,false_mitigation_rate,block_rate_on_attack,escalation_rate
Rule-based,854,146,0,0.0,7,0.008197,0.500000,0.000
DQN,854,146,0,0.0,0,0.000000,0.739726,0.002


DQN: true label × action


action,ALLOW,BLOCK,ESCALATE,MONITOR,RATE_LIMIT
true_label,,,,,
Bots,0,0,0,2,0
Brute Force,0,0,0,3,0
DDoS,0,51,0,0,0
DoS,0,57,1,0,0
Normal Traffic,854,0,0,0,0
Port Scanning,0,0,0,0,31
Web Attacks,0,0,1,0,0


In [37]:
RL_SAMPLE_PLAN = {
    "Normal Traffic": 3000,
    "DoS": 2500,
    "DDoS": 2000,
    "Port Scanning": 1500,
    "Brute Force": 500,
    "Web Attacks": 500,
    "Bots": 500
}

rl_samples = []

for attack_type, sample_size in RL_SAMPLE_PLAN.items():
    class_data = train_df[
        train_df["Attack Type"] == attack_type
    ]

    sampled_class_data = class_data.sample(
        n=sample_size,
        random_state=42
    )

    rl_samples.append(sampled_class_data)

rl_train_df = pd.concat(
    rl_samples,
    ignore_index=True
)

# Xáo trộn để các class không nằm thành từng khối liên tiếp
rl_train_df = rl_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("RL training flows:", len(rl_train_df))
display(rl_train_df["Attack Type"].value_counts())

RL training flows: 10500


Attack Type
Normal Traffic    3000
DoS               2500
DDoS              2000
Port Scanning     1500
Web Attacks        500
Brute Force        500
Bots               500
Name: count, dtype: int64

In [39]:
import json

with open(
    MODELS_DIR / "xgboost_metadata.json",
    "r",
    encoding="utf-8"
) as file:
    xgb_metadata = json.load(file)

feature_columns = xgb_metadata["feature_columns"]

print("Number of features:", len(feature_columns))
print("First 5 features:", feature_columns[:5])

assert len(feature_columns) == 52
assert feature_columns == xgb_model.feature_names_in_.tolist()

Number of features: 52
First 5 features: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max']


In [40]:
from tqdm.auto import tqdm

rl_train_results = []

for row_index in tqdm(
    rl_train_df.index,
    desc="Building RL training results"
):
    flow_features = rl_train_df.loc[
        [row_index],
        feature_columns
    ]

    true_label = rl_train_df.at[
        row_index,
        "Attack Type"
    ]

    result = analyze_flow(
        flow_features,
        true_label=true_label
    )

    rl_train_results.append(result)

print("Number of RL results:", len(rl_train_results))
display(pd.DataFrame(rl_train_results).head())

Building RL training results:   0%|          | 0/10500 [00:00<?, ?it/s]

Number of RL results: 10500


,true_label,anomaly_score,is_anomaly,predicted_attack,confidence,attack_probabilities
0,DoS,3.503445,True,DoS,0.999919,"{'Bots': 1.0421682645755936e-06, 'Brute Force'..."
1,Port Scanning,0.201190,False,Port Scanning,0.999728,"{'Bots': 4.186315891274717e-06, 'Brute Force':..."
2,Port Scanning,0.018632,False,Port Scanning,0.944731,"{'Bots': 1.895568311738316e-05, 'Brute Force':..."
3,DDoS,0.042280,False,DDoS,0.999678,"{'Bots': 3.7666791286028456e-07, 'Brute Force'..."
4,Port Scanning,0.018098,False,Port Scanning,0.968902,"{'Bots': 7.334552265092498e-06, 'Brute Force':..."


In [41]:
rl_train_env = NetworkDefenseEnv(
    rl_train_results
)

initial_state, _ = rl_train_env.reset()

print("RL training flows:", len(rl_train_results))
print("Initial state shape:", initial_state.shape)

assert len(rl_train_results) == 10500
assert initial_state.shape == (11,)

RL training flows: 10500
Initial state shape: (11,)


In [42]:
from stable_baselines3 import DQN

dqn_final_model = DQN(
    policy="MlpPolicy",
    env=rl_train_env,
    learning_rate=5e-4,
    buffer_size=50_000,
    learning_starts=2_000,
    batch_size=128,
    gamma=0.95,
    exploration_fraction=0.25,
    exploration_final_eps=0.05,
    target_update_interval=1_000,
    policy_kwargs={
        "net_arch": [128, 128]
    },
    verbose=1,
    seed=42,
    device="cuda"
)

print("Final DQN device:", dqn_final_model.device)

dqn_final_model.learn(
    total_timesteps=100_000,
    progress_bar=True
)

Output()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Final DQN device: cuda


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.05e+04 |
|    ep_rew_mean      | 5.95e+04 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 914      |
|    time_elapsed     | 45       |
|    total_timesteps  | 42000    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 3.95     |
|    n_updates        | 9999     |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.05e+04 |
|    ep_rew_mean      | 7.6e+04  |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 867      |
|    time_elapsed     | 96       |
|    total_timesteps  | 84000    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 3.14     |
|    n_updates        | 20499    |
----------------------------------


In [43]:
print(
    "Final training timesteps:",
    dqn_final_model.num_timesteps
)

Final training timesteps: 100000


In [44]:
final_dqn_evaluation_env = NetworkDefenseEnv(
    evaluation_results
)

final_dqn_results_1000, final_dqn_total_reward_1000 = (
    run_dqn_policy(
        final_dqn_evaluation_env,
        dqn_final_model
    )
)

final_dqn_metrics = calculate_defense_metrics(
    final_dqn_results_1000
)

comparison_df = pd.DataFrame({
    "Policy": [
        "Rule-based",
        "DQN smoke test",
        "DQN final"
    ],
    "Total reward": [
        rule_total_reward_1000,
        dqn_total_reward_1000,
        final_dqn_total_reward_1000
    ],
    "Average reward": [
        rule_results_1000["reward"].mean(),
        dqn_results_1000["reward"].mean(),
        final_dqn_results_1000["reward"].mean()
    ]
})

display(comparison_df)

display(
    pd.DataFrame(
        [rule_metrics, dqn_metrics, final_dqn_metrics],
        index=[
            "Rule-based",
            "DQN smoke test",
            "DQN final"
        ]
    )
)

,Policy,Total reward,Average reward
0,Rule-based,5753.0,5.753
1,DQN smoke test,6108.0,6.108
2,DQN final,6119.0,6.119


,normal_flows,attack_flows,missed_attacks,missed_attack_rate,false_mitigations,false_mitigation_rate,block_rate_on_attack,escalation_rate
Rule-based,854,146,0,0.0,7,0.008197,0.500000,0.000
DQN smoke test,854,146,0,0.0,0,0.000000,0.739726,0.002
DQN final,854,146,0,0.0,0,0.000000,0.739726,0.006


In [45]:
import json
import numpy as np

DQN_MODEL_PATH = MODELS_DIR / "dqn_policy"
DQN_METADATA_PATH = MODELS_DIR / "dqn_policy_metadata.json"

# Stable-Baselines3 tự thêm đuôi .zip
dqn_final_model.save(
    str(DQN_MODEL_PATH)
)

dqn_metadata = {
    "model_type": "Stable-Baselines3 DQN",
    "model_file": "dqn_policy.zip",
    "state_size": 11,
    "actions": ACTIONS,
    "action_to_id": ACTION_TO_ID,
    "reward_table": REWARD_TABLE,
    "training": {
        "training_flows": len(rl_train_results),
        "training_timesteps": int(
            dqn_final_model.num_timesteps
        ),
        "seed": 42,
        "device": str(dqn_final_model.device),
        "sample_plan": RL_SAMPLE_PLAN
    },
    "evaluation_subset": {
        "flows": len(final_dqn_results_1000),
        "total_reward": float(
            final_dqn_total_reward_1000
        ),
        "average_reward": float(
            final_dqn_results_1000["reward"].mean()
        ),
        "defense_metrics": {
            key: (
                value.item()
                if isinstance(value, np.generic)
                else value
            )
            for key, value in final_dqn_metrics.items()
        }
    }
}

with open(
    DQN_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dqn_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Model saved:", DQN_MODEL_PATH.with_suffix(".zip"))
print("Metadata saved:", DQN_METADATA_PATH)

Model saved: D:\Python project\PROJECT\models\dqn_policy.zip
Metadata saved: D:\Python project\PROJECT\models\dqn_policy_metadata.json


In [46]:
from stable_baselines3 import DQN

loaded_dqn_model = DQN.load(
    str(DQN_MODEL_PATH.with_suffix(".zip")),
    device="cuda"
)

reload_env = NetworkDefenseEnv(
    evaluation_results
)

state, _ = reload_env.reset()

action_id, _ = loaded_dqn_model.predict(
    state,
    deterministic=True
)

print("Loaded DQN device:", loaded_dqn_model.device)
print("State shape:", state.shape)
print("First predicted action:", ID_TO_ACTION[int(action_id)])

assert state.shape == (11,)
assert int(action_id) in ID_TO_ACTION

Loaded DQN device: cuda
State shape: (11,)
First predicted action: ALLOW


In [48]:
print("test_df shape:", test_df.shape)

display(
    test_df["Attack Type"]
    .value_counts()
)

test_df shape: (378089, 53)


Attack Type
Normal Traffic    314235
DoS                29062
DDoS               19202
Port Scanning      13604
Brute Force         1373
Web Attacks          321
Bots                 292
Name: count, dtype: int64

In [49]:
EVAL_SAMPLE_PLAN = {
    "Normal Traffic": 3000,
    "DoS": 2000,
    "DDoS": 2000,
    "Port Scanning": 1500,
    "Brute Force": 800,
    "Bots": 292,
    "Web Attacks": 321
}

evaluation_samples = []

for attack_type, sample_size in EVAL_SAMPLE_PLAN.items():
    class_data = test_df[
        test_df["Attack Type"] == attack_type
    ]

    evaluation_samples.append(
        class_data.sample(
            n=sample_size,
            random_state=42
        )
    )

final_evaluation_df = pd.concat(
    evaluation_samples,
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Final evaluation flows:", len(final_evaluation_df))
display(final_evaluation_df["Attack Type"].value_counts())

Final evaluation flows: 9913


Attack Type
Normal Traffic    3000
DDoS              2000
DoS               2000
Port Scanning     1500
Brute Force        800
Web Attacks        321
Bots               292
Name: count, dtype: int64

In [50]:
xgb_model.set_params(device="cuda")

def analyze_flows_batch(
    flows_df,
    batch_size=2048
):
    x_raw = flows_df[
        feature_columns
    ].copy()

    true_labels = flows_df[
        "Attack Type"
    ].to_numpy()

    # Scaler của sklearn chạy CPU
    x_scaled = scaler.transform(
        x_raw
    ).astype(np.float32)

    # Autoencoder inference theo batch trên GPU
    anomaly_score_batches = []

    with torch.inference_mode():
        for start_index in range(
            0,
            len(x_scaled),
            batch_size
        ):
            end_index = start_index + batch_size

            x_batch = torch.as_tensor(
                x_scaled[start_index:end_index],
                dtype=torch.float32,
                device=device
            )

            reconstructed = ae_model(x_batch)

            batch_scores = torch.mean(
                (x_batch - reconstructed) ** 2,
                dim=1
            )

            anomaly_score_batches.append(
                batch_scores.cpu().numpy()
            )

    anomaly_scores = np.concatenate(
        anomaly_score_batches
    )

    # XGBoost inference toàn batch trên GPU
    probabilities = xgb_model.predict_proba(x_raw)

    predicted_ids = probabilities.argmax(axis=1)

    predicted_labels = label_encoder.inverse_transform(
        predicted_ids
    )

    results = []

    for row_index in range(len(flows_df)):
        probability_dict = {
            label: float(probability)
            for label, probability in zip(
                label_encoder.classes_,
                probabilities[row_index]
            )
        }

        results.append({
            "true_label": true_labels[row_index],
            "anomaly_score": float(
                anomaly_scores[row_index]
            ),
            "is_anomaly": bool(
                anomaly_scores[row_index] > ae_threshold
            ),
            "predicted_attack": predicted_labels[row_index],
            "confidence": float(
                probabilities[
                    row_index,
                    predicted_ids[row_index]
                ]
            ),
            "attack_probabilities": probability_dict
        })

    return results


print(
    "Autoencoder device:",
    next(ae_model.parameters()).device
)

print(
    "XGBoost device:",
    xgb_model.get_params().get("device")
)

Autoencoder device: cuda:0
XGBoost device: cuda


In [51]:
final_evaluation_results = analyze_flows_batch(
    final_evaluation_df,
    batch_size=2048
)

print(
    "Number of evaluation results:",
    len(final_evaluation_results)
)

display(
    pd.DataFrame(
        final_evaluation_results
    ).head()
)

Number of evaluation results: 9913


,true_label,anomaly_score,is_anomaly,predicted_attack,confidence,attack_probabilities
0,DDoS,2.202609,True,DDoS,0.999877,"{'Bots': 3.7148237197470735e-07, 'Brute Force'..."
1,Port Scanning,0.158492,False,Port Scanning,0.974568,"{'Bots': 1.0386554095020983e-05, 'Brute Force'..."
2,Normal Traffic,0.127650,False,Normal Traffic,0.999972,"{'Bots': 1.1018425993825076e-06, 'Brute Force'..."
3,DoS,4.425022,True,DoS,0.999918,"{'Bots': 6.292397074503242e-07, 'Brute Force':..."
4,DDoS,1.638860,True,DDoS,0.999911,"{'Bots': 3.9186713252092886e-07, 'Brute Force'..."


In [52]:
# Rule-based evaluation
large_rule_env = NetworkDefenseEnv(
    final_evaluation_results
)

large_rule_results, large_rule_total_reward = run_policy(
    large_rule_env,
    rule_based_policy
)

# DQN final evaluation
large_dqn_env = NetworkDefenseEnv(
    final_evaluation_results
)

large_dqn_results, large_dqn_total_reward = run_dqn_policy(
    large_dqn_env,
    loaded_dqn_model
)

# Safety metrics
large_rule_metrics = calculate_defense_metrics(
    large_rule_results
)

large_dqn_metrics = calculate_defense_metrics(
    large_dqn_results
)

# Reward comparison
large_comparison_df = pd.DataFrame({
    "Policy": ["Rule-based", "DQN final"],
    "Total reward": [
        large_rule_total_reward,
        large_dqn_total_reward
    ],
    "Average reward": [
        large_rule_results["reward"].mean(),
        large_dqn_results["reward"].mean()
    ]
})

display(large_comparison_df)

# Safety comparison
large_metrics_df = pd.DataFrame(
    [large_rule_metrics, large_dqn_metrics],
    index=["Rule-based", "DQN final"]
)

display(large_metrics_df)

,Policy,Total reward,Average reward
0,Rule-based,78206.0,7.889236
1,DQN final,90986.0,9.178453


,normal_flows,attack_flows,missed_attacks,missed_attack_rate,false_mitigations,false_mitigation_rate,block_rate_on_attack,escalation_rate
Rule-based,3000,6913,93,0.013453,32,0.010667,0.393751,0.000000
DQN final,3000,6913,5,0.000723,3,0.001000,0.580211,0.131242


In [53]:
print("DQN: count theo true label × action")
display(
    pd.crosstab(
        large_dqn_results["true_label"],
        large_dqn_results["action"]
    )
)

print("DQN: tỷ lệ action trong từng class")
display(
    pd.crosstab(
        large_dqn_results["true_label"],
        large_dqn_results["action"],
        normalize="index"
    ).round(3)
)

print("5 missed attacks:")
display(
    large_dqn_results[
        (large_dqn_results["true_label"] != "Normal Traffic") &
        (large_dqn_results["action"] == "ALLOW")
    ][
        ["true_label", "predicted_attack", "confidence", "action"]
    ].head(10)
)

print("3 false mitigations:")
display(
    large_dqn_results[
        (large_dqn_results["true_label"] == "Normal Traffic") &
        (large_dqn_results["action"] != "ALLOW")
    ][
        ["true_label", "predicted_attack", "confidence", "action"]
    ].head(10)
)

DQN: count theo true label × action


action,ALLOW,BLOCK,ESCALATE,MONITOR,RATE_LIMIT
true_label,,,,,
Bots,3,24,178,3,84
Brute Force,0,0,799,1,0
DDoS,0,2000,0,0,0
DoS,0,1987,13,0,0
Normal Traffic,2997,0,2,1,0
Port Scanning,1,0,58,0,1441
Web Attacks,1,0,251,3,66


DQN: tỷ lệ action trong từng class


action,ALLOW,BLOCK,ESCALATE,MONITOR,RATE_LIMIT
true_label,,,,,
Bots,0.010,0.082,0.610,0.010,0.288
Brute Force,0.000,0.000,0.999,0.001,0.000
DDoS,0.000,1.000,0.000,0.000,0.000
DoS,0.000,0.994,0.006,0.000,0.000
Normal Traffic,0.999,0.000,0.001,0.000,0.000
Port Scanning,0.001,0.000,0.039,0.000,0.961
Web Attacks,0.003,0.000,0.782,0.009,0.206


5 missed attacks:


,true_label,predicted_attack,confidence,action
936,Bots,Normal Traffic,0.992951,ALLOW
3956,Bots,Normal Traffic,0.984525,ALLOW
4776,Web Attacks,Normal Traffic,0.970909,ALLOW
5163,Bots,Normal Traffic,0.999928,ALLOW
8681,Port Scanning,Normal Traffic,0.932845,ALLOW


3 false mitigations:


,true_label,predicted_attack,confidence,action
4542,Normal Traffic,Bots,0.548757,ESCALATE
5045,Normal Traffic,Normal Traffic,0.840786,MONITOR
7823,Normal Traffic,Normal Traffic,0.773547,ESCALATE


In [54]:
RISK_LEVELS_V2 = {
    "Normal Traffic": "normal",
    "Bots": "contain",
    "Brute Force": "contain",
    "Port Scanning": "contain",
    "Web Attacks": "review",
    "DoS": "critical",
    "DDoS": "critical"
}

REWARD_TABLE_V2 = {
    "normal": {
        "ALLOW": 5,
        "MONITOR": 3,
        "RATE_LIMIT": -5,
        "BLOCK": -30,
        "ESCALATE": -8
    },
    "contain": {
        "ALLOW": -10,
        "MONITOR": 2,
        "RATE_LIMIT": 10,
        "BLOCK": 3,
        "ESCALATE": -5
    },
    "review": {
        "ALLOW": -15,
        "MONITOR": 2,
        "RATE_LIMIT": 4,
        "BLOCK": 5,
        "ESCALATE": 10
    },
    "critical": {
        "ALLOW": -30,
        "MONITOR": -5,
        "RATE_LIMIT": 5,
        "BLOCK": 15,
        "ESCALATE": 12
    }
}

# compute_reward() và NetworkDefenseEnv dùng hai biến này
RISK_LEVELS = RISK_LEVELS_V2
REWARD_TABLE = REWARD_TABLE_V2

In [55]:
for attack_type in [
    "Normal Traffic",
    "Bots",
    "Brute Force",
    "Port Scanning",
    "Web Attacks",
    "DoS",
    "DDoS"
]:
    rewards = [
        compute_reward(
            attack_type,
            ACTION_TO_ID[action]
        )
        for action in ACTIONS
    ]

    print(attack_type, dict(zip(ACTIONS, rewards)))

Normal Traffic {'ALLOW': 5, 'MONITOR': 3, 'RATE_LIMIT': -5, 'BLOCK': -30, 'ESCALATE': -8}
Bots {'ALLOW': -10, 'MONITOR': 2, 'RATE_LIMIT': 10, 'BLOCK': 3, 'ESCALATE': -5}
Brute Force {'ALLOW': -10, 'MONITOR': 2, 'RATE_LIMIT': 10, 'BLOCK': 3, 'ESCALATE': -5}
Port Scanning {'ALLOW': -10, 'MONITOR': 2, 'RATE_LIMIT': 10, 'BLOCK': 3, 'ESCALATE': -5}
Web Attacks {'ALLOW': -15, 'MONITOR': 2, 'RATE_LIMIT': 4, 'BLOCK': 5, 'ESCALATE': 10}
DoS {'ALLOW': -30, 'MONITOR': -5, 'RATE_LIMIT': 5, 'BLOCK': 15, 'ESCALATE': 12}
DDoS {'ALLOW': -30, 'MONITOR': -5, 'RATE_LIMIT': 5, 'BLOCK': 15, 'ESCALATE': 12}


In [56]:
rl_train_env_v2 = NetworkDefenseEnv(
    rl_train_results
)

dqn_v2_model = DQN(
    policy="MlpPolicy",
    env=rl_train_env_v2,
    learning_rate=5e-4,
    buffer_size=50_000,
    learning_starts=2_000,
    batch_size=128,
    gamma=0.95,
    exploration_fraction=0.25,
    exploration_final_eps=0.05,
    target_update_interval=1_000,
    policy_kwargs={
        "net_arch": [128, 128]
    },
    verbose=1,
    seed=42,
    device="cuda"
)

print("DQN v2 device:", dqn_v2_model.device)

dqn_v2_model.learn(
    total_timesteps=100_000,
    progress_bar=True
)

Output()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DQN v2 device: cuda


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.05e+04 |
|    ep_rew_mean      | 6.62e+04 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 944      |
|    time_elapsed     | 44       |
|    total_timesteps  | 42000    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 3.24     |
|    n_updates        | 9999     |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.05e+04 |
|    ep_rew_mean      | 8.54e+04 |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 876      |
|    time_elapsed     | 95       |
|    total_timesteps  | 84000    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 2.86     |
|    n_updates        | 20499    |
----------------------------------


In [57]:
print(
    "Current timesteps:",
    dqn_v2_model.num_timesteps
)

Current timesteps: 100000


In [58]:
# Rule-based với reward policy v2
rule_env_v2 = NetworkDefenseEnv(
    final_evaluation_results
)

rule_results_v2, rule_total_reward_v2 = run_policy(
    rule_env_v2,
    rule_based_policy
)

# DQN v2
dqn_env_v2 = NetworkDefenseEnv(
    final_evaluation_results
)

dqn_results_v2, dqn_total_reward_v2 = run_dqn_policy(
    dqn_env_v2,
    dqn_v2_model
)

rule_metrics_v2 = calculate_defense_metrics(
    rule_results_v2
)

dqn_metrics_v2 = calculate_defense_metrics(
    dqn_results_v2
)

display(pd.DataFrame({
    "Policy": ["Rule-based v2", "DQN v2"],
    "Total reward": [
        rule_total_reward_v2,
        dqn_total_reward_v2
    ],
    "Average reward": [
        rule_results_v2["reward"].mean(),
        dqn_results_v2["reward"].mean()
    ]
}))

display(
    pd.DataFrame(
        [rule_metrics_v2, dqn_metrics_v2],
        index=["Rule-based v2", "DQN v2"]
    )
)

,Policy,Total reward,Average reward
0,Rule-based v2,87420.0,8.818723
1,DQN v2,103924.0,10.483607


,normal_flows,attack_flows,missed_attacks,missed_attack_rate,false_mitigations,false_mitigation_rate,block_rate_on_attack,escalation_rate
Rule-based v2,3000,6913,93,0.013453,32,0.010667,0.393751,0.000000
DQN v2,3000,6913,5,0.000723,4,0.001333,0.576450,0.033491


In [59]:
display(
    pd.crosstab(
        dqn_results_v2["true_label"],
        dqn_results_v2["action"]
    )
)

action,ALLOW,BLOCK,ESCALATE,MONITOR,RATE_LIMIT
true_label,,,,,
Bots,3,0,0,1,288
Brute Force,0,0,0,1,799
DDoS,0,2000,0,0,0
DoS,0,1985,15,0,0
Normal Traffic,2996,0,0,3,1
Port Scanning,1,0,0,0,1499
Web Attacks,1,0,317,3,0


In [60]:
DQN_V2_MODEL_PATH = MODELS_DIR / "dqn_policy_v2"
DQN_V2_METADATA_PATH = (
    MODELS_DIR / "dqn_policy_v2_metadata.json"
)

dqn_v2_model.save(
    str(DQN_V2_MODEL_PATH)
)

dqn_v2_metadata = {
    "model_type": "Stable-Baselines3 DQN",
    "version": "v2",
    "model_file": "dqn_policy_v2.zip",
    "state_size": 11,
    "actions": ACTIONS,
    "action_to_id": ACTION_TO_ID,
    "risk_levels": RISK_LEVELS_V2,
    "reward_table": REWARD_TABLE_V2,
    "training": {
        "training_flows": len(rl_train_results),
        "training_timesteps": int(
            dqn_v2_model.num_timesteps
        ),
        "seed": 42,
        "device": str(dqn_v2_model.device),
        "sample_plan": RL_SAMPLE_PLAN
    },
    "held_out_evaluation": {
        "flows": len(dqn_results_v2),
        "sample_plan": EVAL_SAMPLE_PLAN,
        "total_reward": float(
            dqn_total_reward_v2
        ),
        "average_reward": float(
            dqn_results_v2["reward"].mean()
        ),
        "defense_metrics": {
            key: (
                value.item()
                if isinstance(value, np.generic)
                else value
            )
            for key, value in dqn_metrics_v2.items()
        }
    }
}

with open(
    DQN_V2_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dqn_v2_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

print(
    "DQN v2 model:",
    DQN_V2_MODEL_PATH.with_suffix(".zip")
)

print(
    "DQN v2 metadata:",
    DQN_V2_METADATA_PATH
)

DQN v2 model: D:\Python project\PROJECT\models\dqn_policy_v2.zip
DQN v2 metadata: D:\Python project\PROJECT\models\dqn_policy_v2_metadata.json


In [61]:
dqn_v2_reloaded = DQN.load(
    str(DQN_V2_MODEL_PATH.with_suffix(".zip")),
    device="cuda"
)

reload_env_v2 = NetworkDefenseEnv(
    final_evaluation_results
)

state, _ = reload_env_v2.reset()

action_id, _ = dqn_v2_reloaded.predict(
    state,
    deterministic=True
)

print("Reloaded model device:", dqn_v2_reloaded.device)
print("State shape:", state.shape)
print("First action:", ID_TO_ACTION[int(action_id)])

assert state.shape == (11,)
assert int(action_id) in ID_TO_ACTION

Reloaded model device: cuda
State shape: (11,)
First action: BLOCK


## DQN Summary

Notebook này xây dựng lớp phản ứng tự động sau khi Autoencoder và XGBoost đã phân tích network flow.

Pipeline:

52 features → Autoencoder anomaly score + XGBoost probabilities → state 11 chiều → DQN v2 → response action.

Các action gồm: `ALLOW`, `MONITOR`, `RATE_LIMIT`, `BLOCK`, `ESCALATE`.

Đã thực hiện:

1. Xây dựng state 11 chiều từ anomaly score, 7 xác suất XGBoost và context.
2. Tạo `NetworkDefenseEnv` và reward policy.
3. So sánh rule-based policy với DQN.
4. Phát hiện DQN v1 escalate quá nhiều Brute Force/Bots.
5. Thiết kế reward v2:
   - Normal Traffic → ALLOW
   - Bots/Brute Force/Port Scanning → RATE_LIMIT
   - DoS/DDoS → BLOCK
   - Web Attacks → ESCALATE
6. Train, evaluate, save và reload `dqn_policy_v2.zip`.

Cách xử lý khi output không mong muốn:

- Missed attack cao → tăng phạt `ALLOW` cho attack, kiểm tra lại XGBoost/Autoencoder.
- False mitigation cao → tăng phạt action can thiệp lên Normal Traffic.
- Escalate quá nhiều → giảm reward hoặc tăng phạt `ESCALATE` cho nhóm tương ứng.
- DQN reward thấp hơn rule-based → kiểm tra bảng `true label × action`, reward table và train thêm timesteps.
- Action không hợp lý theo từng lớp → tách class thành risk group riêng và thiết kế lại reward.

Lưu ý: tập 9,913 flow đã được dùng để quan sát rồi điều chỉnh từ v1 sang v2, nên nên gọi nó là validation-like evaluation subset. Kết quả cuối cùng cho CV cần chạy một lần trên test subset chưa từng dùng để tuning.